In [10]:
import pandas as pd
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import dump
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df = pd.read_excel('./student_synthetic_housing_data.xlsx')  # <<< updated for Excel

# Features and target
features = [
    'Age', 'Adults', 'Children', 'Rent', 'Distance_to_New_Tenancy', 'Total_Rooms', 'Area_m2',
    'Hospital_distance', 'Gym_distance', 'School_distance',
    'Supermarket_distance', 'Distance_to_University'
]
X = df[features]
y = df['Label']

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

xgb_model = Pipeline(
    steps=[
        ("regressor", XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)),
    ]
)

lgbm_model = Pipeline(
    steps=[
        (
            "regressor",
            LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=-1, random_state = 42),
        ),
    ]
)

# Stacking ensemble
stacked_model = StackingRegressor(
    estimators=[ ("xgb", xgb_model), ("lightgbm", lgbm_model)],
    final_estimator=LinearRegression(),
    cv=5,
)

stacked_model.fit(X_train, y_train)

y_pred = stacked_model.predict(X_val)

dump(stacked_model, 'student_bolig_recommendation_model.joblib')
# Evaluation
mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002136 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2072
[LightGBM] [Info] Number of data points in the train set: 17805, number of used features: 12
[LightGBM] [Info] Start training from score 0.741433
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001574 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2072
[LightGBM] [Info] Number of data points in the train set: 14244, number of used features: 12
[LightGBM] [Info] Start training from score 0.741409
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001364 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2072
[LightGBM] [Info] Number of data points in the train set: 14244, number of used features: 12
[LightGBM] [Info] Start tra

In [11]:
# Save the model
dump(stacked_model, '../student_bolig_recommendation_model.joblib')

print(f"Mean Absolute Error: {mae:.2f}")
print(f"Mean Squared Error: {mse:.2f}")
print("R² Score:", r2)

Mean Absolute Error: 0.02
Mean Squared Error: 0.00
R² Score: 0.9894920261933415
